In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import geopandas as gpd
import zipfile
import re
import os

In [ ]:
dir = '/content/drive/MyDrive/Projects/Transit Data Challenge/Dataset/ttc-subway-delay-data-2021.xlsx'

df = pd.read_excel(dir)

In [ ]:
df.head()

,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,2021-01-01,00:33,Friday,BLOOR STATION,MUPAA,0,0,N,YU,6046
1,2021-01-01,00:39,Friday,SHERBOURNE STATION,EUCO,5,9,E,BD,5250
2,2021-01-01,01:07,Friday,KENNEDY BD STATION,EUCD,5,9,E,BD,5249
3,2021-01-01,01:41,Friday,ST CLAIR STATION,MUIS,0,0,NaN,YU,0
4,2021-01-01,02:04,Friday,SHEPPARD WEST STATION,MUIS,0,0,NaN,YU,0


In [ ]:
xls = pd.ExcelFile(dir)
print(xls.sheet_names)

['January21', 'Feb 21', "March '21", "April '21", "May '21", 'June 21', 'July 21', 'August 21', 'Sept 21', 'Oct 21', 'Nov 21', 'December21']


In [ ]:
# read every sheet
all_sheets = []

for sheet in xls.sheet_names:

    temp_df = pd.read_excel(dir, sheet_name=sheet)

    # optional: add source month
    temp_df['month_sheet'] = sheet

    all_sheets.append(temp_df)

# combine into one dataframe
df = pd.concat(all_sheets, ignore_index=True)

print(df.shape)
df.head()

(16370, 11)


,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle,month_sheet
0,2021-01-01,00:33,Friday,BLOOR STATION,MUPAA,0,0,N,YU,6046,January21
1,2021-01-01,00:39,Friday,SHERBOURNE STATION,EUCO,5,9,E,BD,5250,January21
2,2021-01-01,01:07,Friday,KENNEDY BD STATION,EUCD,5,9,E,BD,5249,January21
3,2021-01-01,01:41,Friday,ST CLAIR STATION,MUIS,0,0,NaN,YU,0,January21
4,2021-01-01,02:04,Friday,SHEPPARD WEST STATION,MUIS,0,0,NaN,YU,0,January21


In [ ]:
print(df.columns.tolist())

['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle', 'month_sheet']


In [ ]:
line_mapping = {
    'YU': 1,
    'BD': 2,
    'SRT': 3,
    'SHP': 4
}

# normalize
df['Line'] = (
    df['Line']
    .astype(str)
    .str.upper()
    .str.strip()
)

# split entries like YU/BD
df['line_list'] = df['Line'].str.split('/')

# explode rows
df = df.explode('line_list')

df['line_list'] = df['line_list'].str.strip()

# map to route ids
df['route_id'] = df['line_list'].map(line_mapping)

# remove invalid
df = df[df['route_id'].notna()]

In [ ]:
def clean_station(x):
    x = str(x).upper().strip()
    x = re.sub(r'\s+STATION$', '', x)
    x = re.sub(r'\s+', ' ', x)
    return x

df['station_clean'] = df['Station'].apply(clean_station)

def norm(x):
    x = str(x).upper().strip()
    x = re.sub(r'[\.\-]', ' ', x)
    x = re.sub(r'\s+', ' ', x)
    return x

df['station_key'] = df['station_clean'].apply(norm)

In [ ]:
# gtfs stations
gtfs_dir = '/content/drive/MyDrive/Projects/Transit Data Challenge/Dataset/Complete GTFS'

stops = pd.read_csv(f"{gtfs_dir}/stops.txt", low_memory=False)
stop_times = pd.read_csv(f"{gtfs_dir}/stop_times.txt", low_memory=False)
trips = pd.read_csv(f"{gtfs_dir}/trips.txt", low_memory=False)

def clean_gtfs_station(x):
    x = str(x).upper().strip()
    x = re.sub(r'\s*-\s*(NORTHBOUND|SOUTHBOUND|EASTBOUND|WESTBOUND).*', '', x)
    x = re.sub(r'\s*PLATFORM.*', '', x)
    x = re.sub(r'\s*-\s*SUBWAY.*', '', x)
    x = re.sub(r'\s+STATION$', '', x)
    x = re.sub(r'\s+', ' ', x)
    return x.strip()

stops['station_clean'] = stops['stop_name'].apply(clean_gtfs_station)
stops['station_key'] = stops['station_clean'].apply(norm)

trips['trip_id'] = trips['trip_id'].astype(str)
stop_times['trip_id'] = stop_times['trip_id'].astype(str)

trip_routes = trips[['trip_id', 'route_id']].drop_duplicates()
stop_routes = stop_times.merge(trip_routes, on='trip_id', how='left')
route_stops = stop_routes[['route_id', 'stop_id']].drop_duplicates()

stations = (
    route_stops
    .merge(
        stops[['stop_id', 'station_clean', 'stop_lat', 'stop_lon']],
        on='stop_id',
        how='left'
    )
)

stations['stop_lat'] = stations['stop_lat'].astype(float)
stations['stop_lon'] = stations['stop_lon'].astype(float)

stations = stations.dropna(subset=['stop_lat', 'stop_lon'])
stations = stations.drop_duplicates(subset=['route_id', 'station_clean'])

In [ ]:
zip_path  = '/content/drive/MyDrive/Projects/Transit Data Challenge/Dataset/ttc-subway-shapefile-wgs84.zip'

shapes_raw = pd.read_csv(f"{gtfs_dir}/shapes.txt", dtype=str)
shapes_raw['shape_pt_lat']      = shapes_raw['shape_pt_lat'].astype(float)
shapes_raw['shape_pt_lon']      = shapes_raw['shape_pt_lon'].astype(float)
shapes_raw['shape_pt_sequence'] = shapes_raw['shape_pt_sequence'].astype(int)

trips_all = pd.read_csv(f"{gtfs_dir}/trips.txt", low_memory=False)

In [ ]:
def get_longest_shape(route_id):
    shape_ids = trips_all[trips_all['route_id'] == route_id]['shape_id'].unique()
    return max(shape_ids, key=lambda sid: len(shapes_raw[shapes_raw['shape_id'] == sid]))

route_polylines = {}
for route_id in [1, 2, 4]:
    shape_id = get_longest_shape(route_id)
    pts = shapes_raw[shapes_raw['shape_id'] == shape_id].sort_values('shape_pt_sequence')
    route_polylines[route_id] = list(zip(pts['shape_pt_lat'], pts['shape_pt_lon']))

LINE3_STATIONS = [
    {'name': 'Kennedy Station',            'lat': 43.73205491519861, 'lon': -79.26473171879161},
    {'name': 'Lawrence East Station',      'lat': 43.75090042148507, 'lon': -79.27044776112001},
    {'name': 'Ellesmere Station',          'lat': 43.76708403355233, 'lon': -79.27742831637770},
    {'name': 'Midland Station',            'lat': 43.77059611255155, 'lon': -79.27214037830714},
    {'name': 'Scarborough Centre Station', 'lat': 43.77450825133060, 'lon': -79.25791334941826},
    {'name': 'McCowan Station',            'lat': 43.77513248991619, 'lon': -79.25183956111863},
]
route_polylines[3] = [(s['lat'], s['lon']) for s in LINE3_STATIONS]

In [ ]:
station_coords = {}

for _, row in stations.iterrows():
    key = (norm(row['station_clean']), int(row['route_id']))
    station_coords[key] = (row['stop_lat'], row['stop_lon'])

line3_coords = {
    (norm(s['name'].upper().replace(' STATION', '')), 3): (s['lat'], s['lon'])
    for s in LINE3_STATIONS
}
station_coords.update(line3_coords)
print(f"Loaded {len(station_coords)} station records")

Loaded 14041 station records


In [ ]:
# mismatch station names
delay_keys   = set(zip(df['station_key'], df['route_id'].astype(int)))
coord_keys   = set(station_coords.keys())

unmatched = [(s, r) for s, r in delay_keys if (s, r) not in coord_keys]
unmatched.sort(key=lambda x: x[1])

for station, route in unmatched:
    print(f"Line {route}: '{station}'")

Line 1: 'ST GEORGE YU'
Line 1: 'SUMMERHILL TO BLOOR ST'
Line 1: 'SPADINA STATION TO KIN'
Line 1: 'GUNN BUILDING'
Line 1: 'YONGE / UNIVERSITY B'
Line 1: 'YONGE UNIVERSITY BLOOR'
Line 1: 'SHEPPARD WEST WILSON'
Line 1: 'ST CLAIR WEST TO KING'
Line 1: 'VMC STATION TO PIONEER'
Line 1: 'FINCH TO SHEPPARD STAT'
Line 1: 'LINE 1'
Line 1: 'KING STATION TO ST GEO'
Line 1: 'SPADINA TO KING STATIO'
Line 1: 'WILSON DIVISION 2ND'
Line 1: 'DUNDAS'
Line 1: 'QUEEN'S QUAY'
Line 1: 'YOUNGE UNIVERSITY SPAD'
Line 1: 'YONGE UNIVERSTIY SUBWA'
Line 1: 'EGLINTON STATION APPRO'
Line 1: 'ST GEORGE YUS'
Line 1: 'SHEPPARD WEST TO WILSO'
Line 1: 'YONGE UNIVERSITY SUBWA'
Line 1: 'UNION STATION (TO KING'
Line 1: 'SHEPPARD WYE'
Line 1: 'YONGE/UNIVERSITY LINE'
Line 1: 'NORTH YORK CTR'
Line 1: 'YONGE AND STEELES'
Line 1: 'ZONE 2'
Line 1: 'YONGE'
Line 1: 'UNION TO KING'
Line 1: 'FINCH STATION TO ST CL'
Line 1: 'ST CLAIR WEST TO ST AN'
Line 1: 'EGLINTON STATION (ENTE'
Line 1: 'FINCH WEST TO LAWRENCE'
Line 1: 'SHEPPARD WEST

DATA QUALITY NOTES -- TTC Delay Dataset 2021
-------------------------------------------------------
The Station field in the raw delay data is free-text entered by TTC operators,
which causes several classes of mismatch against GTFS station names:

1. TRUNCATION: Excel's cell character limit cuts off long entries
   e.g. 'EGLINTON STATION TO VA' instead of 'EGLINTON STATION TO VAUGHAN MC'
   These are stretch/segment descriptions anyway, not true station names.

2. TYPOS & ABBREVIATIONS: Inconsistent operator shorthand
   e.g. 'VAUGHAN MC' vs 'VAUGHAN METROPOLITAN CENTRE', 'KILPING' vs 'KIPLING',
   'OSSIGNTON' vs 'OSSINGTON', 'LAWERENCE' vs 'LAWRENCE'
   Partially corrected via NAME_FIXES mapping.

3. SEGMENT DESCRIPTIONS logged as station names — the most common case
   e.g. 'KING STATION TO ST GEORGE', 'WILSON TO SHEPPARD WEST', 'FINCH TO ROSEDALE'
   These represent delays that occurred between stations and cannot be
   pinned to a single location. Dropped from the heatmap.

4. WRONG LINE: A small number of entries appear under the wrong route
   e.g. 'LESLIE' and 'DON MILLS' logged under Line 2 (Bloor-Danforth)
   when those stations only exist on Line 4 (Sheppard). Likely operator error.
   These will not match any GTFS coordinate and are dropped.

5. NON-STATION LOCATIONS: Yards, carhouses, divisions, and control centres
   e.g. 'WILSON CARHOUSE', 'GREENWOOD YARD', 'TRANSIT CONTROL CENTRE'
   These are legitimate TTC facilities but have no map coordinates in GTFS.
   Dropped from the heatmap as they are operational, not passenger-facing.

NET EFFECT: The heatmap reflects only delays that could be confidently
geo-located to a specific station. Unmatched entries are dropped, meaning
total delay counts per station are likely an undercount of true delays.

In [ ]:
name_changes = {
    # line 1
    'VAUGHAN MC':             'VAUGHAN METROPOLITAN CENTRE',
    'NORTH YORK CTR':         'NORTH YORK CENTRE',
    'QUEENS PARK':            "QUEEN'S PARK",
    'LAWERENCE':              'LAWRENCE',
    'EGLINTON WEST':          'EGLINTON',
    'DUNDAS WEST':            'DUNDAS WEST',
    'PIONEER VILLAGE STATIO': 'PIONEER VILLAGE',
    'YORK UNIVERSITY STATIO': 'YORK UNIVERSITY',
    'SHEPPARD':               'SHEPPARD YONGE',
    'YONGE AND SHEPPARD':     'SHEPPARD YONGE',
    'YONGE AND BLOOR':        'BLOOR',
    'VICTORIA PARK':          None,

    # line 2
    'KENNEDY BD':             'KENNEDY',
    'OSSIGNTON':              'OSSINGTON',
    'KILPING':                'KIPLING',
    'LESLIE':                 'LESLIE',
    'DON MILLS':              'DON MILLS',
    'SCARBOROUGH CTR STATIO': 'SCARBOROUGH CENTRE',

    # line 3
    'KENNEDY SRT':            'KENNEDY',
    'SCARBOROUGH CTR STATIO': 'SCARBOROUGH CENTRE',

    # line 4
    'YONGE AND SHEPPARD':     'SHEPPARD YONGE',
    'SHEPPARD':               'SHEPPARD YONGE',
}

df['station_key'] = df['station_key'].replace(name_changes)
df = df[df['station_key'].notna()]

In [ ]:
def get_coords(row):
    return station_coords.get(
        (row['station_key'], int(row['route_id'])),
        (np.nan, np.nan)
    )

In [ ]:
# aggregate delays per station
delay_agg = (
    df.groupby(['route_id', 'station_key'])
    .agg(
        total_delay=('Min Delay', 'sum'),
        incident_count=('Min Delay', 'count'),
        avg_delay=('Min Delay', 'mean')
    )
    .reset_index()
)

delay_agg[['lat', 'lon']] = delay_agg.apply(
    lambda r: pd.Series(get_coords(r)),
    axis=1
)

delay_agg = delay_agg.dropna(subset=['lat', 'lon'])


# normalize each of the 3 delay factors
delay_agg['norm_total']    = delay_agg['total_delay']    / delay_agg['total_delay'].max()
delay_agg['norm_count']    = delay_agg['incident_count'] / delay_agg['incident_count'].max()
delay_agg['norm_avg']      = delay_agg['avg_delay']      / delay_agg['avg_delay'].max()

# weighted composite (can adjust weights as needed)
W_TOTAL = (1/3)
W_COUNT = (1/3)
W_AVG   = (1/3)

delay_agg['scaled_heat'] = (
    W_TOTAL * delay_agg['norm_total'] +
    W_COUNT * delay_agg['norm_count'] +
    W_AVG   * delay_agg['norm_avg']
) * 10

heat_data = delay_agg[['lat', 'lon', 'scaled_heat']].values.tolist()

In [ ]:
LINE_CONFIG = {
    1: {'name': 'Line 1 · Yonge-University', 'color': '#FFCD00'},
    2: {'name': 'Line 2 · Bloor-Danforth',   'color': '#00A650'},
    3: {'name': 'Line 3 · Scarborough RT',   'color': '#0054A6'},
    4: {'name': 'Line 4 · Sheppard',         'color': '#B000B0'},
}

m = folium.Map(location=[43.700, -79.420], zoom_start=12, tiles=None, control_scale=True)
folium.TileLayer('CartoDB positron', name='', show=True, control=False).add_to(m)

for route_id, cfg in LINE_CONFIG.items():
    if route_id not in route_polylines:
        continue
    layer = folium.FeatureGroup(name=cfg['name'], show=True)
    folium.PolyLine(
        locations=route_polylines[route_id],
        color=cfg['color'],
        weight=5,
        opacity=0.75 if route_id == 3 else 0.85,
        tooltip=cfg['name'] + (' (Closed 2023)' if route_id == 3 else ''),
    ).add_to(layer)
    layer.add_to(m)

heat_layer = folium.FeatureGroup(name='Delay Heatmap', show=True)
HeatMap(
    heat_data,
    radius=35,
    blur=25,
    max_zoom=13,
    min_opacity=0.4,
    gradient={
        0.2: '#ffffb2',
        0.4: '#fecc5c',
        0.6: '#fd8d3c',
        0.8: '#f03b20',
        1.0: '#bd0026',
    }
).add_to(heat_layer)
heat_layer.add_to(m)

delay_layer = folium.FeatureGroup(name='Delay Stations', show=True)

for _, row in delay_agg.iterrows():
    line_color = LINE_CONFIG.get(int(row['route_id']), {}).get('color', '#888888')
    tooltip = (
        f"<b>{row['station_key']}</b><br>"
        f"Total Delay: {row['total_delay']:.0f} min<br>"
        f"Incidents: {row['incident_count']}<br>"
        f"Avg Delay: {row['avg_delay']:.1f} min"
    )
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(4, np.log1p(row['incident_count']) * 2),
        color='white',
        weight=1,
        fill=True,
        fill_color=line_color,
        fill_opacity=0.8,
        tooltip=tooltip,
    ).add_to(delay_layer)

delay_layer.add_to(m)

In [ ]:
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;background:white;border:1px solid #ccc;
border-radius:8px;padding:12px 16px;font-family:Arial,sans-serif;font-size:13px;
z-index:9999;box-shadow:2px 2px 6px rgba(0,0,0,0.2)">
  <b style='font-size:14px'>TTC Subway Delays 2021</b><br/><br/>
  <span style='color:#FFCD00'>&#9644;</span> Line 1 · Yonge-University<br/>
  <span style='color:#00A650'>&#9644;</span> Line 2 · Bloor-Danforth<br/>
  <span style='color:#0054A6'>&#9644;</span> Line 3 · Scarborough RT <i>(Closed 2023)</i><br/>
  <span style='color:#B000B0'>&#9644;</span> Line 4 · Sheppard<br/>
  <br/>
  <span style='background:linear-gradient(to right,#ffffb2,#fecc5c,#fd8d3c,#f03b20,#bd0026);
    display:inline-block;width:100px;height:10px;border-radius:3px;vertical-align:middle'></span>
  &nbsp;Low → High delay severity<br/>
  <br/>
  <small>
    <b>Heatmap</b> = weighted composite score<br/>
    &nbsp;&nbsp;· Total delay minutes (33.3%)<br/>
    &nbsp;&nbsp;· Incident frequency (33.3%)<br/>
    &nbsp;&nbsp;· Avg delay per incident (33.3%)<br/>
    <br/>
    <b>Circles</b> = per-station stats (hover for details)<br/>
    &nbsp;&nbsp;· Size = incident count<br/>
    &nbsp;&nbsp;· Colour = line
  </small>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False, position='topright').add_to(m)
display(m)